In [ ]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
from nltk.cluster.util import cosine_distance
import numpy as np
import networkx as nx
nltk.download('punkt')
nltk.download('stopwords')
with open("MTG_Rules.txt", 'r', encoding='utf-8') as file:
        text = file.read()
# Preprocess the text
sentences = sent_tokenize(text)
stop_words = set(stopwords.words('english'))
def preprocess_sentence(sentence):
    words = word_tokenize(sentence.lower())
    words = [word for word in words if word.isalnum() and word not in stop_words]
    return words
# Sentence scoring based on term frequency
def score_sentences(sentences):
    sentence_scores = []
    word_frequencies = FreqDist([word for sentence in sentences for word in
    preprocess_sentence(sentence)])
    for sentence in sentences:
        words = preprocess_sentence(sentence)
        sentence_score = sum(word_frequencies[word] for word in words)
        sentence_scores.append((sentence, sentence_score))
    return sentence_scores
# Select top-ranked sentences
def select_sentences(sentence_scores, num_sentences=2):
    sentence_scores.sort(key=lambda x: x[1], reverse=True)
    selected_sentences = [sentence[0] for sentence in
    sentence_scores[:num_sentences]]
    return selected_sentences
# Generate summary
sentence_scores = score_sentences(sentences)
summary_sentences = select_sentences(sentence_scores)
summary = ' '.join(summary_sentences)
print("Summary:")
print(summary)


[nltk_data] Downloading package punkt to /home/jacob/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/jacob/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Summary:
It means “You may designate two legendary cards as your commander rather than one if each has a ‘partner with [name]’ ability with the other’s name” and “When this permanent enters, target player may search their library for a card named [name], reveal it, put it into their hand, then shuffle.”

702.124k “Choose a Background” means “You may designate two cards as your commander rather than one if one of them is this card and the other is a legendary Background enchantment card.” You can’t designate two cards as your commander if one has a “choose a Background” ability and the other is not a legendary Background enchantment card, and legendary Background enchantment cards can’t be your commander unless you have also designated a commander with “choose a Background.”

702.124m “Doctor’s companion” means “You may designate two legendary creature cards as your commander rather than one if one of them is this card and the other is a legendary Time Lord Doctor creature card that has

In [5]:
# Reference from text - Extractive
from rouge import Rouge
ROUGE = Rouge()

reference = "The mana symbols are {W}, {U}, {B}, {R}, {G}, and {C}; the numerical symbols {0}, {1}, {2}, {3}, {4}, and so on; the variable symbol {X}; the hybrid symbols {W/U}, {W/B}, {U/B}, {U/R}, {B/R}, {B/G}, {R/G}, {R/W}, {G/W}, and {G/U}; the monocolored hybrid symbols {2/W}, {2/U}, {2/B}, {2/R}, {2/G}, {C/W}, {C/U}, {C/B}, {C/R}, and {C/G}; the Phyrexian mana symbols {W/P}, {U/P}, {B/P}, {R/P}, and {G/P}; the hybrid Phyrexian symbols {W/U/P}, {W/B/P}, {U/B/P}, {U/R/P}, {B/R/P}, {B/G/P}, {R/G/P}, {R/W/P}, {G/W/P}, and {G/U/P}; and the snow mana symbol {S}."
candidate = "There are many different mana symbols in Magic: The Gathering. Mana symbols can be symbolic singularly or in combinations such as {W}, {W/B}, and numerical symbols may have numbers or X for a variable cost such as {2}, {3/W}, or {X/R} "
print(ROUGE.get_scores(candidate, reference))


[{'rouge-1': {'r': 0.15873015873015872, 'p': 0.29411764705882354, 'f': 0.20618556245722192}, 'rouge-2': {'r': 0.024390243902439025, 'p': 0.05128205128205128, 'f': 0.03305784687111594}, 'rouge-l': {'r': 0.1111111111111111, 'p': 0.20588235294117646, 'f': 0.14432989235412916}}]


In [4]:
# Reference from text - Abstractive
from rouge import Rouge
ROUGE = Rouge()

reference = "This document is the ultimate authority for Magic: The Gathering® game play. It consists of a series of numbered rules followed by a glossary. Many of the numbered rules are divided into subrules, and each separate rule and subrule of the game has its own number. Changes may have been made to this document since its publication. You can download the most recent version from the Magic rules website at Magic.Wizards.com/Rules."
candidate = "This document dictates the rules for Magic: The Gathering. Rules are organized by number and category and may contain subrules. This document may change as more new mechanics are added, and has already been changed. You can always find the most up to date rules on the website."
print(ROUGE.get_scores(candidate, reference))

[{'rouge-1': {'r': 0.3333333333333333, 'p': 0.48717948717948717, 'f': 0.39583332850911457}, 'rouge-2': {'r': 0.07142857142857142, 'p': 0.10869565217391304, 'f': 0.08620689176575531}, 'rouge-l': {'r': 0.3157894736842105, 'p': 0.46153846153846156, 'f': 0.37499999517578125}}]
